In [102]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier



import nltk
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [103]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [104]:
rows = df.loc[df.duplicated(subset=['article', 'title'])]

#print(df.duplicated(subset=['article']).sum())
#print(df.duplicated(subset=['title']).sum())
#print(df.duplicated(subset=['title','article']).sum())

duplicated = df.loc[df.duplicated(subset=['title','article'])]

mask = df['article'].isin(duplicated['article'])
to_drop = df[mask]

df.drop(to_drop["Id"], inplace=True)
df.dropna(inplace=True)

In [ ]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [106]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
text = df['title'] + ' ' + df['title'] + ' ' + df['article']
df = pd.concat([df, text], axis=1)

#rinomino le colonne e droppo title e article
df.rename(columns={0: 'text'}, inplace=True)
df.drop(['title', 'article'], inplace=True, axis=1)

df.columns

Index(['Id', 'source', 'page_rank', 'timestamp', 'text'], dtype='object')

In [107]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y, shuffle=True)

In [108]:
import numpy as np
import pandas as pd

def process_dates(df, col_name='date'):
    # 1. 'coerce': Se trovi "0000-00-00", trasformalo in NaT (Null) invece di crashare
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce')
    
    # 2. Estrai i componenti base
    # Usiamo .dt direttamente. Dove la data è NaT, otterremo NaN (che va bene per XGBoost!)
    df['hour'] = df[col_name].dt.hour
    df['month'] = df[col_name].dt.month
    df['day_of_week'] = df[col_name].dt.dayofweek
    
    # 3. Cyclical Encoding (Seno e Coseno)
    # Se 'hour' è NaN, anche il seno/coseno sarà NaN.
    # XGBoost gestisce i NaN automaticamente imparando come trattarli.
    
    # ORA
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    
    # MESE
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    
    # GIORNO
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)
    
    # Opzionale: Riempiamo i NaN con 0 per pulizia (0 è il centro del cerchio, "neutro")
    # Oppure lasciamoli NaN, XGBoost se la cava. Per sicurezza riempiamoli con 0.
    new_cols = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_sin', 'day_cos']
    df[new_cols] = df[new_cols].fillna(0)
    
    return df

# --- RILANCIA ---
X_train = process_dates(X_train, col_name='timestamp') # O il nome corretto della colonna
X_test = process_dates(X_test, col_name='timestamp')

In [109]:
#ora faccio ohe sulla colonna source, scalo page_rank e vettorizzo la colonna text
time_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_sin', 'day_cos']


encoder = OneHotEncoder(min_frequency=1, handle_unknown='infrequent_if_exist')
scaler = StandardScaler()
text_pipeline = Pipeline([
    ('vect', TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        ngram_range=(1, 2),
        max_df=0.6,
        min_df=5
    )),
    ('svd', TruncatedSVD(n_components=300, algorithm='arpack', random_state=RANDOM_SEED)) 
])

vectorizer1 = TfidfVectorizer(
        max_features=50000,
        stop_words='english',
        ngram_range=(1, 2),
        max_df=0.9,
        min_df=3,
        strip_accents='unicode')

preprocessor = ColumnTransformer(transformers=[
    ('source', encoder, ['source']),
    ('page_rank', scaler, ['page_rank']),
    ('text',vectorizer1, 'text'),
    ('time', 'passthrough', time_features)
], remainder='drop', n_jobs=-1)


#    ('page_rank', scaler, ['page_rank']),

In [ ]:
TOP_K_PER_CLASS = 7000 

final_vocabulary = set() 
print("Estrazione vocabolario specifico per classe...")
classes = np.unique(y_train)

for label in classes:
    subset_text = X_train[y_train == label]['text']
    
    temp_vectorizer = TfidfVectorizer(
        max_features=TOP_K_PER_CLASS,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2
    )
    temp_vectorizer.fit(subset_text)    
    
    words = temp_vectorizer.get_feature_names_out()
    final_vocabulary.update(words)
    print(f"Classe {label}: trovate {len(words)} parole specifiche.")

final_vocabulary_list = list(final_vocabulary)
print(f"Vocabolario totale ottimizzato: {len(final_vocabulary_list)} parole uniche.")


master_vectorizer = TfidfVectorizer(
    vocabulary=final_vocabulary_list,  
    stop_words=stop_words
)

preprocessor_custom = ColumnTransformer(transformers=[
    ('source', encoder, ['source']),
    ('text', master_vectorizer, 'text'), 
    ('time', 'passthrough', time_features)
], remainder='drop', n_jobs=-1)

Estrazione vocabolario specifico per classe...
Classe 0: trovate 7000 parole specifiche.
Classe 1: trovate 7000 parole specifiche.
Classe 2: trovate 7000 parole specifiche.
Classe 3: trovate 7000 parole specifiche.
Classe 4: trovate 7000 parole specifiche.
Classe 5: trovate 7000 parole specifiche.
Classe 6: trovate 7000 parole specifiche.
Vocabolario totale ottimizzato: 23159 parole uniche.


In [111]:
#weights = compute_sample_weight(class_weight='balanced', y=y_train)
#
#
## Questo è l'equivalente "veloce" di un SVM lineare
## loss='hinge' -> Si comporta come un SVM
## loss='modified_huber' -> Si comporta come SVM ma ti dà anche le probabilità (predict_proba)
#svm_fast = SGDClassifier(
#    loss='modified_huber', # Consiglio questo se vuoi fare Ensemble dopo
#    penalty='l2',
#    alpha=1e-4,            # Regolarizzazione standard
#    random_state=42,
#    max_iter=3000,
#    class_weight='balanced', # <--- Eccolo qui, il tuo amico per le classi rare
#    n_jobs=-1
#)
#
## Definisci il modello XGBoost
#clf = XGBClassifier(
#    n_estimators=1000,      # Numero di alberi
#    learning_rate=0.1,     # Impara piano per essere preciso
#    max_depth=6,            # Profondità degli alberi
#    subsample=0.8,          # Usa solo l'80% dei dati per ogni albero (evita overfitting)
#    colsample_bytree=0.8,   # Usa solo l'80% delle feature per albero
#    n_jobs=-1,              # Usa tutti i core
#    random_state=42,
#    tree_method="hist"      # Molto più veloce su dataset grandi
#    # Non serve class_weight='balanced', XGBoost gestisce meglio.
#    # Se vuoi forzarlo, usa scale_pos_weight ma è complesso per multiclasse.
#)
#
#X_train = preprocessor_custom.fit_transform(X_train)
#
## Fit
#svm_fast.fit(X_train, y_train)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = svm_fast.predict(X_test)
#
#
#print(classification_report(y_test, y_pred))

In [112]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
weights = compute_sample_weight(class_weight='balanced', y=y_train)
# 1. Definisci il Primo Generale: XGBoost (Il tuo modello attuale)
# Nota: Ho mantenuto i parametri che funzionavano bene
xgb_clf = XGBClassifier(
    n_estimators=1000,
    max_depth=6,            # O 8, se la GridSearch ti ha detto 8
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

# 2. Definisci il Secondo Generale: SVM Veloce (SGD)
# loss='modified_huber' è vitale: permette all'SVM di calcolare probabilità
svm_clf = SGDClassifier(
    loss='modified_huber', 
    penalty='l2',
    alpha=1e-4,
    max_iter=2000,
    tol=1e-3,
    random_state=42,
    n_jobs=-1
    # NON mettiamo class_weight='balanced' qui perché passeremo i pesi globali dopo
)

# 3. Crea il Consiglio di Guerra (VotingClassifier)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', xgb_clf), 
        ('svm', svm_clf)
    ],
    voting='soft',   # 'soft' = Media delle probabilità (MOLTO meglio di 'hard')
    weights=[1.2, 1] # Diamo un leggero bonus (20%) a XGBoost perché sappiamo che è forte
)

# 4. Inseriscilo nella Pipeline
# Usa il tuo preprocessor migliore (quello con 50k feature o quello custom)
final_ensemble_pipeline = Pipeline([
    ('preprocessor', preprocessor_custom), # O preprocessor_custom
    ('ensemble', voting_clf)
])

# 5. ADDESTRAMENTO
print("⚔️ Inizio addestramento dell'Ensemble (XGB + SVM)...")
# Passiamo i pesi (sample_weight) all'ensemble. 
# Lui li passerà automaticamente sia a XGBoost che a SVM.
final_ensemble_pipeline.fit(X_train, y_train, ensemble__sample_weight=weights)

# 6. VALUTAZIONE
print("Valutazione finale...")
y_pred_ensemble = final_ensemble_pipeline.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_ensemble))

⚔️ Inizio addestramento dell'Ensemble (XGB + SVM)...
Valutazione finale...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

              precision    recall  f1-score   support

           0       0.83      0.69      0.76      4438
           1       0.75      0.85      0.80      1848
           2       0.85      0.84      0.84      2119
           3       0.61      0.59      0.60      1778
           4       0.82      0.95      0.88      1663
           5       0.52      0.55      0.53      2117
           6       0.61      0.86      0.71       554

    accuracy                           0.74     14517
   macro avg       0.71      0.76      0.73     14517
weighted avg       0.74      0.74      0.73     14517



In [113]:
#X_train = preprocessor_custom.fit_transform(X_train)
#clf = LogisticRegression(random_state=RANDOM_SEED,C=1, penalty='l2', solver='saga', class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = clf.predict(X_test)
#
#print(classification_report(y_test, y_pred))

In [114]:
y_train_pred = clf.predict(X_train)
print(classification_report(y_train, y_train_pred))


NotFittedError: need to call fit or load_model beforehand